(course-enzyme-conformational-engineering)=
# Module 27: Conformational Engineering

Explore a sidechain orientation in the PET hydrolase structure
[6EQE](https://www.rcsb.org/structure/6EQE). You will rotate **TRP, chain A,
group ID 159** using its explicit chi1 quartet. This is a controlled geometric
perturbation; it does not establish a functional gate, substrate accessibility,
rotamer stability or an energy minimum.

> **🎯 Learning Outcomes**
> - Identify a sidechain by deposited chain and group labels.
> - Measure chi1 and chi2 using atom-index quartets.
> - Rotate one covalent block while preserving the source.

In [1]:
import numpy as np
import molsysmt as msm
from molsysmt import pyunitwizard as puw

molsys = msm.convert(msm.systems['PETase']['6eqe.pdb'], to_form='molsysmt.MolSys')
original_coordinates = puw.get_value(msm.get(molsys, coordinates=True), to_unit='nm').copy()
site = 'chain_id=="A" and group_id=="159" and group_name=="TRP"'
quartet = np.asarray(msm.topology.get_dihedral_quartets(molsys, selection=site, chi1=True))[0]
chi2_quartet = np.asarray(msm.topology.get_dihedral_quartets(molsys, selection=site, chi2=True))[0]

(course-enzyme-conformational-engineering-reading)=
## 1. Measuring the Sidechain

Use the offline original PDB with its deposited identifiers. Normal PDB conversion
completes recognized standard-group bonds; this is separate from hydrogen placement
or chemical-state preparation. No hydrogen preparation is requested in this example.

In [2]:
before = msm.structure.get_dihedral_angles(molsys, dihedral_quartets=quartet, pbc=False)
chi2 = msm.structure.get_dihedral_angles(molsys, dihedral_quartets=chi2_quartet, pbc=False)
print('TRP A159 chi1 (degrees):', puw.get_value(before, to_unit='degrees'))
print('TRP A159 chi2 (degrees):', puw.get_value(chi2, to_unit='degrees'))

TRP A159 chi1 (degrees): [[177.72124091]]
TRP A159 chi2 (degrees): [[38.30538473]]


(course-enzyme-conformational-engineering-rotation)=
## 2. Rotating an Explicit Block

Cut the CA-CB central bond and select the component containing the fourth atom.
An independent result retains all other coordinates, topology and box. It has
chi1 shifted by 180 degrees, understood modulo 360 degrees.

In [3]:
components = msm.topology.get_covalent_blocks(
    molsys, remove_bonds=quartet[1:3], output_type='sets')
moving = sorted(next(block for block in components if int(quartet[3]) in block))
edited = msm.structure.shift_dihedral_angles(
    molsys, dihedral_quartets=quartet, shifts='180 degrees',
    blocks=[moving], pbc=False)
after = msm.structure.get_dihedral_angles(edited, dihedral_quartets=quartet, pbc=False)
print('Edited chi1 (degrees):', puw.get_value(after, to_unit='degrees'))

Edited chi1 (degrees): [[-2.27875909]]


(course-enzyme-conformational-engineering-inspection)=
## 3. Inspecting the Result

Compare the original and edited coordinates before assigning a physical role to
the perturbation. You can inspect the neighborhood using
`msm.view(edited, selection='within 0.8 nm of (chain_id=="A" and group_id=="159")')`;
see the [visualization tutorial](../../user/tools/basic/view.ipynb).
No visualization or energetic validation is required to measure the achieved angle.

(course-enzyme-conformational-engineering-challenge)=
## 🏆 Challenge: The Dihedral Master

Find SER A160, discover its chi1 quartet, set a target of -60 degrees and verify
it with {func}`molsysmt.structure.get_dihedral_angles`. Treat this as an imposed
rotamer; assess clashes and stability separately.

:::{key-takeaway}
:class: dropdown
Explicit quartets identify the measured torsion; explicit moving blocks identify
what an edit moves. A verified angle is a geometric result, not physical validation.
:::

(course-enzyme-conformational-engineering-see-also)=
:::{seealso}
:class: dropdown
{func}`molsysmt.topology.get_dihedral_quartets`,
{func}`molsysmt.topology.get_covalent_blocks`,
{func}`molsysmt.structure.shift_dihedral_angles`,
and [Module 16: Structural Operations](../Common_Core/16_Structural_Operations.ipynb).
:::